# Model Exploration Notes

This notebook documents findings about how OLMo 3 7B Think behaves through
Inspect-ai's local HuggingFace provider. Unlike the data exploration notebook,
this one isn't meant to be re-run, it's a record of things learned
and why the pipeline (Notebook 3) is built the way it is.

In [ ]:
!pip install -q inspect-ai transformers accelerate torch


In [ ]:
import glob
import json
import transformers
from inspect_ai.log import read_eval_log
from transformers import AutoTokenizer, AutoConfig

## Smoke Test: Does Inspect-ai's HF provider correctly load a specific checkpoint?

We confirmed from Inspect-ai's source code that `revision=` should pass through
to `from_pretrained()`, but this has never been tested against a real checkpoint
through the actual Inspect-ai CLI. This is that test: load step_0025 specifically,
not just the model in general, and run one trivial sample.

In [ ]:
%%writefile smoke_test.py
from inspect_ai import Task, task
from inspect_ai.dataset import Sample
from inspect_ai.solver import generate
from inspect_ai.scorer import match
from inspect_ai.model import GenerateConfig

@task
def smoke_test():
    return Task(
        dataset=[Sample(input="What is 2+2? Answer with just the number.", target="4")],
        solver=[generate()],
        scorer=match(),
        config=GenerateConfig(
            temperature=0.6,
            top_p=0.95,
            max_tokens=32768
        )
    )

Overwriting smoke_test.py


### Run with default settings (expected to crash)

Running the eval without any special flags first. OLMo's chat template assumes
dict-style messages but Inspect-ai passes Python objects. This will crash.
The error is captured in the log and read in the cells below.

In [ ]:
!inspect eval smoke_test.py --model hf/allenai/Olmo-3-7B-Think -M revision=step_0025 --limit 1 --log-level debug

In [ ]:
log = read_eval_log("logs/2026-07-15T16-54-15-00-00_smoke-test_h7magiBmCvSYVeYHYcBfHL.eval")

print("Status:", log.status)
print("\n--- Samples ---")
for sample in log.samples:
    print("Input:", sample.input)
    print("Target:", sample.target)
    print("Output:", sample.output.completion if sample.output else None)
    print("Score:", sample.scores)

Status: error

--- Samples ---
Input: What is 2+2? Answer with just the number.
Target: 4
Output: 
Score: {}


### **Error captured in log:**

message="'inspect_ai.model._chat_message.ChatMessageUser object' has no attribute 'get'" traceback='Traceback (most recent call last)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("allenai/Olmo-3-7B-Think")
print(tokenizer.chat_template)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{% set has_system = messages|selectattr('role', 'equalto', 'system')|list|length > 0 %}{% if not has_system %}{{ '<|im_start|>system
You are OLMo, a helpful function-calling AI assistant built by Ai2. Your date cutoff is November 2024, and your model weights are available at https://huggingface.co/allenai. You do not currently have access to any functions. <functions></functions><|im_end|>
' }}{% endif %}{% for message in messages %}{% if message['role'] == 'system' %}{{ '<|im_start|>system
' + message['content'] }}{% if message.get('functions', none) is not none %}{{ ' <functions>' + message['functions'] + '</functions><|im_end|>
' }}{% else %}{{ ' You do not currently have access to any functions. <functions></functions><|im_end|>
' }}{% endif %}{% elif message['role'] == 'user' %}{% if message.get('functions', none) is not none %}{{ '<|im_start|>user
' + message['content'] + '
' + '<functions>' + message['functions'] + '</functions><|im_end|>
' }}{% else %}{{ '<|im_start|>user
' + m

## Finding: OLMo's chat template assumes dict-style messages

The smoke test above failed because OLMo's chat_template calls `.get()` on each
message, assuming a plain dict. Inspect-ai passes ChatMessage objects instead,
which don't have `.get()`.
Notably, OLMo's own "Think" model card recommends plain string tokenization for inference - the chat template exists for structured deployment use cases, not for the kind of direct model evaluation being done here.

Trying `use_chat_template=False`, which makes Inspect-ai fall back to its own
plain-text formatting (role: content) instead of OLMo's Jinja template.

In [ ]:
!inspect eval smoke_test.py --model hf/allenai/Olmo-3-7B-Think -M revision=step_0025 -M use_chat_template=False --limit 1

In [ ]:
# grab the most recent log file
latest_log = sorted(glob.glob("logs/*.eval"))[-1]
log = read_eval_log(latest_log)

print("Status:", log.status)
for sample in log.samples:
    print("Input:", sample.input)
    print("Target:", sample.target)
    print("Output:", sample.output.completion if sample.output else None)
    print("Score:", sample.scores)

Status: success
Input: What is 2+2? Answer with just the number.
Target: 4
Output: <think>
First, the user asked: "What is 2+2?" and specified to answer with just the number. So, I need to provide a numerical answer without any additional text.

The expression is 2 + 2. Addition is straightforward. 2 plus 2 equals 4.

I should ensure that my response is only the number, as per the instruction. No explanations, no words, just the result.

As an AI, I'm supposed to be helpful and harmless. In this case, since the user wants a direct answer, I shouldn't overcomplicate it. But internally, I know that 2 + 2 is a basic arithmetic operation that equals 4.

Possible reasons for the simplicity: The user might be testing if I can follow instructions precisely, or it could be a very young user learning addition.

My response must be concise. Just "4" should suffice.

I recall that in some contexts, like binary, 2+2 might be different, but the user didn't specify any base. It's standard to assume 

In [ ]:
log = read_eval_log(sorted(glob.glob("logs/*.eval"))[-1])

# Confirm GenerateConfig was applied
cfg = log.plan.config
print(f"temperature: {cfg.temperature}")
print(f"top_p: {cfg.top_p}")
print(f"max_tokens: {cfg.max_tokens}")

# How many tokens were actually used
usage = log.samples[0].output.usage
print(f"\nTokens used this run:")
print(f"  input: {usage.input_tokens}")
print(f"  output: {usage.output_tokens}")
print(f"  total: {usage.total_tokens}")

temperature: 0.6
top_p: 0.95
max_tokens: 32768

Tokens used this run:
  input: 15
  output: 449
  total: 464


### Confirming what Inspect-ai actually sends to the model

Checking `sample.messages` to verify no hidden system message is being injected.
Only a `[user]` message should appear — the model's `[assistant]` response follows.

In [ ]:
log = read_eval_log(sorted(glob.glob("logs/*.eval"))[-1])
for sample in log.samples:
    print("=== MESSAGES SENT TO MODEL ===")
    for msg in sample.messages:
        print(f"[{msg.role}]: {msg.content}")

=== MESSAGES SENT TO MODEL ===
[user]: What is 2+2? Answer with just the number.
[assistant]: <think>
First, the user asked: "What is 2+2?" and specified to answer with just the number. So, I need to provide a numerical answer without any additional text.

The expression is 2 + 2. Addition is straightforward. 2 plus 2 equals 4.

I should ensure that my response is only the number, as per the instruction. No explanations, no words, just the result.

As an AI, I'm supposed to be helpful and harmless. In this case, since the user wants a direct answer, I shouldn't overcomplicate it. But internally, I know that 2 + 2 is a basic arithmetic operation that equals 4.

Possible reasons for the simplicity: The user might be testing if I can follow instructions precisely, or it could be a very young user learning addition.

My response must be concise. Just "4" should suffice.

I recall that in some contexts, like binary, 2+2 might be different, but the user didn't specify any base. It's standard

## Generation Settings

To ensure consistent, reproducible measurements across all 14 checkpoints, we need
explicit generation settings rather than relying on Inspect-ai's defaults (which are
`None`, deferring to whatever transformers decides). OLMo's architecture supports
up to 65,536 tokens total; the model card recommends `max_tokens=32768` for generation.

The Task definition in `smoke_test.py` uses:
```python
config=GenerateConfig(temperature=0.6, top_p=0.95, max_tokens=32768)
```
The cells below confirm what Inspect-ai's defaults are and what OLMo's architecture limit is.



In [ ]:
from inspect_ai.model import GenerateConfig
import inspect
print(inspect.getsource(GenerateConfig))

class GenerateConfig(BaseModel):
    """Model generation options."""

    max_retries: int | None = Field(default=None)
    """Maximum number of times to retry request (defaults to unlimited)."""

    timeout: int | None = Field(default=None)
    """Timeout (in seconds) for an entire request (including retries)."""

    attempt_timeout: int | None = Field(default=None)
    """Timeout (in seconds) for any given attempt (if exceeded, will abandon attempt and retry according to max_retries)."""

    max_connections: int | None = Field(default=None)
    """Maximum number of concurrent connections to Model API (default is model specific)."""

    adaptive_connections: bool | int | AdaptiveConcurrency | None = Field(default=None)
    """Adaptive concurrency for model API connections. Defaults to enabled (`None` and `True` both resolve to `AdaptiveConcurrency()` defaults: min=10, start=20, max=100). Pass `False` to opt out (uses static concurrency). Pass an integer `N` as shorthand for `Adapt

In [ ]:
# Inspect-ai's max_tokens defaults to None ("model specific" — unpredictable)
# We explicitly set max_tokens=32768 in smoke_test.py (confirmed in eval output above)
# OLMo's architecture ceiling:
print("max_tokens default:", GenerateConfig().max_tokens)  # None = defer to transformers

max_tokens default: None


### OLMo's architecture limit


In [ ]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained("allenai/Olmo-3-7B-Think", revision="step_0025")
# Hard ceiling: input + output tokens combined cannot exceed this
print("max_position_embeddings:", config.max_position_embeddings)
# 65536 total, with max_tokens=32768 for output, we have plenty of headroom for long EA inputs

max_position_embeddings: 65536


## `<think>` Extraction

The model's full output (thinking trace + final answer) lives in `sample.output.completion`.
For Notebook 3, every scorer needs to split this into two parts:
- the thinking trace (between `<think>` and `</think>`) - used for EA and IC scoring
- the final answer (after `</think>`) - used for MATH scoring

This cell confirms the extraction logic works and handles the case where `<think>` is absent.



In [ ]:
log = read_eval_log(sorted(glob.glob("logs/*.eval"))[-1])
sample = log.samples[0]
completion = sample.output.completion

if "<think>" in completion and "</think>" in completion:
    thinking = completion.split("<think>")[1].split("</think>")[0].strip()
    answer = completion.split("</think>")[1].strip()
else:
    thinking = None
    answer = completion.strip()

print("=== THINKING TRACE ===")
print(thinking if thinking else "None — no <think> tags found")
print("\n=== FINAL ANSWER ===")
print(answer)
print("\n=== Has thinking trace? ===", thinking is not None)

=== THINKING TRACE ===
First, the user asked: "What is 2+2?" and specified to answer with just the number. So, I need to provide a numerical answer without any additional text.

The expression is 2 + 2. Addition is straightforward. 2 plus 2 equals 4.

I should ensure that my response is only the number, as per the instruction. No explanations, no words, just the result.

As an AI, I'm supposed to be helpful and harmless. In this case, since the user wants a direct answer, I shouldn't overcomplicate it. But internally, I know that 2 + 2 is a basic arithmetic operation that equals 4.

Possible reasons for the simplicity: The user might be testing if I can follow instructions precisely, or it could be a very young user learning addition.

My response must be concise. Just "4" should suffice.

I recall that in some contexts, like binary, 2+2 might be different, but the user didn't specify any base. It's standard to assume base 10 unless stated otherwise.

The question is "2+2", which is cl

## Summary: What We Know Before Building Notebook 3

### Pipeline approach — confirmed working
- **Model loading:** `hf/allenai/Olmo-3-7B-Think` with `-M revision=step_XXXX` correctly
  loads specific RLVR training checkpoints through Inspect-ai's HF provider. Confirmed
  from actual source code: `revision` passes through `**model_args` to `from_pretrained()`.
- **Chat template:** OLMo's Jinja template assumes dict-style messages but Inspect-ai
  passes `ChatMessageUser` objects — crashes with `UndefinedError: has no attribute 'get'`.
  Fix: `-M use_chat_template=False`. This matches OLMo's own recommended inference
  approach (plain string tokenization per the model card).
- **Generation settings:** use `GenerateConfig(temperature=0.6, top_p=0.95, max_tokens=32768)`
  matching the model card's recommended settings. OLMo's architecture limit is 65,536
  tokens; 32,768 for output leaves sufficient headroom for long EA transcripts.

### Prompt structure — confirmed
- Inspect-ai sends only a `[user]` message — no system message injected.
  Confirmed via `sample.messages`: `[user]: prompt`, `[assistant]: completion`.
- The model references "the system prompt" in its reasoning because it was trained
  with system messages during SFT — this is baked into the weights, not something
  we're sending. Expected behavior, not a bug.

### Output structure — confirmed
- Model self-initiates `<think>` reasoning without any priming. The full output in
  `sample.output.completion` is: `<think>...reasoning...</think>\nfinal answer`
- Extraction logic for Notebook 3:
```python
if "<think>" in completion and "</think>" in completion:
    thinking = completion.split("<think>")[1].split("</think>")[0].strip()
    answer = completion.split("</think>")[1].strip()
else:
    thinking = None
    answer = completion.strip()
```
- `thinking` → used for EA and IC scoring (full reasoning trace, no truncation)
- `answer` → used for MATH scoring

### Token usage
- Simple smoke test: 15 input tokens, 449 output tokens (varies by run due to sampling).
- Real tasks (EA transcripts, IC scenarios, MATH) will use substantially more.
  `max_tokens=32768` provides safe headroom.

### What Notebook 3 needs
Every Task in Notebook 3 should use:
- `--model hf/allenai/Olmo-3-7B-Think -M revision=step_XXXX -M use_chat_template=False`
- `config=GenerateConfig(temperature=0.6, top_p=0.95, max_tokens=32768)`
- The extraction logic above in every custom Scorer